In [11]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# --- configuration (match Fig5/01_peak_snapshots.ipynb quiver settings) ---
TIMES_TO_PLOT_OBS = [
    "20250923 2100",
    "20250923 2200",
    "20250923 2300",
]

QUIVER_SCALE = 500
QUIVER_WIDTH = 0.0050
QUIVER_HEADWIDTH = 3.0
QUIVER_COLOR = "red"

FIGSIZE = (6, 6)
DPI = 600

# in-figure quiver notation
QUIVERKEY_U = 30
QUIVERKEY_LABEL = rf"{QUIVERKEY_U} m s$^{{-1}}$"
QUIVERKEY_FONTSIZE = 18
QUIVERKEY_X = 0.08
QUIVERKEY_Y = 0.93

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 18


def _find_fig5_dirs() -> tuple[Path, Path]:
    """Return (data_dir, publish_dir) for Fig5.

    Robust to different CWDs (workspace root, Figs root, or Fig5).
    """
    cwd = Path.cwd().resolve()

    if (cwd / "data" / "mean_wspd_obs.csv").exists():
        fig5_dir = cwd
    elif (cwd / "Fig5" / "data" / "mean_wspd_obs.csv").exists():
        fig5_dir = cwd / "Fig5"
    else:
        parent_with_fig5 = None
        for p in [cwd, *cwd.parents]:
            if (p / "Fig5" / "data" / "mean_wspd_obs.csv").exists():
                parent_with_fig5 = p
                break
        if parent_with_fig5 is None:
            raise FileNotFoundError(
                "Could not find Fig5/data/mean_wspd_obs.csv from current working directory. ",
                f"CWD={cwd}",
            )
        fig5_dir = parent_with_fig5 / "Fig5"

    data_dir = fig5_dir / "data"
    publish_dir = fig5_dir / "publish"
    publish_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, publish_dir


def _find_station_location_xlsx() -> Path:
    """Locate Fig4/data/weather_station_location.xlsx robustly."""
    cwd = Path.cwd().resolve()

    for candidate in [
        cwd / "Fig4" / "data" / "weather_station_location.xlsx",
        cwd / "Figs" / "Fig4" / "data" / "weather_station_location.xlsx",
    ]:
        if candidate.exists():
            return candidate

    for p in [cwd, *cwd.parents]:
        candidate = p / "Fig4" / "data" / "weather_station_location.xlsx"
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not locate Fig4/data/weather_station_location.xlsx from CWD={cwd}")


def _choose_station_sheet_and_columns(xlsx_path: Path, station_codes: list[str]) -> tuple[str, str, str, str]:
    """Heuristically pick (sheet, station_col, lat_col, lon_col).

    Works even if column headers are not English (e.g., 经度/纬度) by detecting
    lat/lon columns from plausible numeric ranges.
    """

    station_set = set([s.strip().lower() for s in station_codes])

    xl = pd.ExcelFile(xlsx_path)
    best_pick: tuple[int, int, str, str, str, str] | None = None
    # (valid_rows, station_matches, sheet, station_col, lat_col, lon_col)

    for sheet in xl.sheet_names:
        df = xl.parse(sheet_name=sheet)
        if df.empty:
            continue

        cols = list(df.columns)

        # 1) station column: maximize number of matches to station_set
        station_col = None
        station_matches = 0
        for c in cols:
            st = df[c].astype(str).str.strip().str.lower()
            m = int(st.isin(station_set).sum())
            if m > station_matches:
                station_matches = m
                station_col = c

        if station_col is None or station_matches == 0:
            continue

        st = df[station_col].astype(str).str.strip().str.lower()
        st_ok = st.isin(station_set)

        # 2) coordinate columns: detect by numeric ranges
        numeric = {}
        lat_scores: list[tuple[int, str]] = []
        lon_scores: list[tuple[int, str]] = []

        for c in cols:
            s = pd.to_numeric(df[c], errors="coerce")
            numeric[c] = s
            lat_cnt = int(s.between(20, 30).sum())
            lon_cnt = int(s.between(110, 120).sum())
            if lat_cnt > 0:
                lat_scores.append((lat_cnt, c))
            if lon_cnt > 0:
                lon_scores.append((lon_cnt, c))

        if not lat_scores or not lon_scores:
            continue

        lat_scores.sort(reverse=True)
        lon_scores.sort(reverse=True)

        # Try top candidates; keep small for speed/robustness
        top_lat = [c for _, c in lat_scores[:5]]
        top_lon = [c for _, c in lon_scores[:5]]

        for lat_col in top_lat:
            lat = numeric[lat_col]
            lat_ok = lat.between(20, 30)
            for lon_col in top_lon:
                if lon_col == lat_col:
                    continue
                lon = numeric[lon_col]
                lon_ok = lon.between(110, 120)

                valid_rows = int((st_ok & lat_ok & lon_ok).sum())
                if valid_rows < 5:
                    continue

                pick = (valid_rows, station_matches, sheet, station_col, lat_col, lon_col)
                if best_pick is None or pick[:2] > best_pick[:2]:
                    best_pick = pick

    if best_pick is None:
        preview = {s: list(pd.ExcelFile(xlsx_path).parse(s, nrows=1).columns) for s in xl.sheet_names}
        raise ValueError(
            "Could not auto-detect station/lat/lon columns in station location xlsx. ",
            f"Sheets/columns preview: {preview}",
        )

    _, _, sheet, station_col, lat_col, lon_col = best_pick
    return sheet, station_col, lat_col, lon_col


def _winddir_to_uv(wspd: np.ndarray, wdir_deg_from: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Convert (speed, direction-from) to (u, v) in m/s.

    Convention: direction is FROM which the wind blows, in degrees clockwise from north.
    """
    theta = np.deg2rad(wdir_deg_from)
    u = -wspd * np.sin(theta)
    v = -wspd * np.cos(theta)
    return u, v


# --- 1) locate files ---
data_dir, publish_dir = _find_fig5_dirs()
wspd_path = data_dir / "mean_wspd_obs.csv"
wdir_path = data_dir / "mean_wdir_obs.csv"
station_xlsx = _find_station_location_xlsx()

print(f"Using data_dir: {data_dir}")
print(f"Using publish_dir: {publish_dir}")
print(f"Using station xlsx: {station_xlsx}")


# --- 2) load obs wind speed/direction ---
df_wspd = pd.read_csv(wspd_path, dtype={"timestamp_utc": str})
df_wdir = pd.read_csv(wdir_path, dtype={"timestamp_utc": str})

for df in (df_wspd, df_wdir):
    df["timestamp_utc"] = df["timestamp_utc"].astype(str).str.strip()

want = set(TIMES_TO_PLOT_OBS)
sub_wspd = df_wspd[df_wspd["timestamp_utc"].isin(want)].copy()
sub_wdir = df_wdir[df_wdir["timestamp_utc"].isin(want)].copy()

sub_wspd = sub_wspd.set_index("timestamp_utc").loc[TIMES_TO_PLOT_OBS]
sub_wdir = sub_wdir.set_index("timestamp_utc").loc[TIMES_TO_PLOT_OBS]

station_cols = [c for c in sub_wspd.columns if c in set(sub_wdir.columns)]
if len(station_cols) == 0:
    raise ValueError("No common station columns between mean_wspd_obs and mean_wdir_obs")

sub_wspd = sub_wspd[station_cols]
sub_wdir = sub_wdir[station_cols]

print("Extracted station wspd (m/s) at target times:")
display(sub_wspd)
print("Extracted station wdir (deg, from) at target times:")
display(sub_wdir)


# --- 3) load station lat/lon metadata ---
sheet, station_col, lat_col, lon_col = _choose_station_sheet_and_columns(station_xlsx, station_cols)
print(f"Selected xlsx sheet={sheet!r}, station_col={station_col!r}, lat_col={lat_col!r}, lon_col={lon_col!r}")

df_loc = pd.read_excel(station_xlsx, sheet_name=sheet)
df_loc = df_loc.rename(columns={station_col: "station", lat_col: "lat", lon_col: "lon"}).copy()

df_loc["station"] = df_loc["station"].astype(str).str.strip().str.lower()

station_cols_norm = [s.strip().lower() for s in station_cols]
loc = df_loc[df_loc["station"].isin(set(station_cols_norm))].copy()
loc["lat"] = pd.to_numeric(loc["lat"], errors="coerce")
loc["lon"] = pd.to_numeric(loc["lon"], errors="coerce")
loc = loc.dropna(subset=["lat", "lon"]).copy()

loc = loc.set_index("station")
keep_order = [s for s in station_cols_norm if s in loc.index]
loc = loc.loc[keep_order].reset_index()

missing_coords = sorted(list(set(station_cols_norm) - set(loc["station"])))
if missing_coords:
    print(f"Stations missing coordinates and will be skipped: {missing_coords}")

print(f"Stations with coordinates: {len(loc)}")
display(loc.head())


# --- 4) compute common map extent (consistent framing across 3 times) ---
if len(loc) < 3:
    raise ValueError("Too few stations with coordinates to plot")

pad = 0.08  # degrees
extent = [
    float(loc["lon"].min()) - pad,
    float(loc["lon"].max()) + pad,
    float(loc["lat"].min()) - pad,
    float(loc["lat"].max()) + pad,
]


def _plot_station_quiver(time_tag: str, wspd_row: pd.Series, wdir_row: pd.Series, out_path: Path):
    # Align to loc order
    wspd = pd.to_numeric(wspd_row.reindex(loc["station"].values), errors="coerce").to_numpy(dtype=float)
    wdir = pd.to_numeric(wdir_row.reindex(loc["station"].values), errors="coerce").to_numpy(dtype=float)

    # Treat 0-speed+0-dir as missing (common encoding)
    missing = (np.isclose(wspd, 0.0) & np.isclose(wdir, 0.0)) | np.isnan(wspd) | np.isnan(wdir)

    u, v = _winddir_to_uv(wspd, wdir)
    u = u.astype(float)
    v = v.astype(float)
    u[missing] = np.nan
    v[missing] = np.nan

    fig = plt.figure(figsize=FIGSIZE)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # land/ocean background
    ax.add_feature(cfeature.OCEAN.with_scale("10m"))
    ax.add_feature(cfeature.LAND.with_scale("10m"))
    # ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.7)

    q = ax.quiver(
        loc["lon"].to_numpy(),
        loc["lat"].to_numpy(),
        u,
        v,
        color=QUIVER_COLOR,
        scale=QUIVER_SCALE,
        width=QUIVER_WIDTH,
        headwidth=QUIVER_HEADWIDTH,
        transform=ccrs.PlateCarree(),
    )

    ax.quiverkey(
        q,
        X=QUIVERKEY_X,
        Y=QUIVERKEY_Y,
        U=QUIVERKEY_U,
        label=QUIVERKEY_LABEL,
        labelpos="E",
        coordinates="axes",
        fontproperties={"family": "Arial", "size": QUIVERKEY_FONTSIZE},
    )

    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


# --- 5) generate and save figures ---
for t in TIMES_TO_PLOT_OBS:
    out_path = publish_dir / f"wind_obs_stations_{t.replace(' ', '_')}.tif"
    _plot_station_quiver(t, sub_wspd.loc[t], sub_wdir.loc[t], out_path)

print("Done.")


Using data_dir: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig5\data
Using publish_dir: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig5\publish
Using station xlsx: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig4\data\weather_station_location.xlsx
Extracted station wspd (m/s) at target times:


,cch,cp1,gi,hka,hko,hks,jkb,kp,lfs,ngp,...,skg,slw,tkl,tme,tms,tmt,tpk,wgl,wlp,yts
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
20250923 2100,24.5,21.5,24.6,19.7,17.6,9.9,14.4,8.6,19.2,27.9,...,19.4,9.2,10.5,28.2,25.3,13.6,16.0,32.8,3.6,32.9
20250923 2200,28.3,15.1,29.0,21.2,18.4,8.7,9.7,14.7,15.2,32.1,...,15.7,12.9,10.9,30.5,26.6,19.4,21.0,37.1,8.1,31.5
20250923 2300,27.2,24.7,35.0,18.2,17.6,12.1,9.0,9.4,16.8,35.1,...,27.2,16.6,7.9,33.6,31.3,19.7,17.8,37.8,8.5,33.3


Extracted station wdir (deg, from) at target times:


,cch,cp1,gi,hka,hko,hks,jkb,kp,lfs,ngp,...,skg,slw,tkl,tme,tms,tmt,tpk,wgl,wlp,yts
timestamp_utc,,,,,,,,,,,,,,,,,,,,,
20250923 2100,45.0,91.0,46.0,21.0,62.0,53.0,42.0,48.0,35.0,66.0,...,61.0,37.0,59.0,75.0,69.0,43.0,65.0,53.0,20.0,26.0
20250923 2200,52.0,99.0,50.0,43.0,64.0,68.0,44.0,58.0,65.0,76.0,...,48.0,60.0,58.0,89.0,77.0,57.0,86.0,63.0,57.0,64.0
20250923 2300,70.0,85.0,59.0,54.0,91.0,85.0,120.0,68.0,61.0,67.0,...,53.0,76.0,78.0,89.0,87.0,73.0,80.0,69.0,60.0,83.0


Selected xlsx sheet='Sheet1', station_col='Unnamed: 2', lat_col='Unnamed: 3', lon_col='Unnamed: 4'
Stations with coordinates: 30


,station,处理1分钟平均风向、1分钟平均风速和1分钟最大风速,Unnamed: 1,lat,lon,Unnamed: 5
0,cch,NaN,11.0,22.201111,114.026667,99
1,cp1,NaN,21.0,22.288889,114.155833,30
2,gi,NaN,9.0,22.285000,114.112778,107
3,hka,NaN,2.0,22.309444,113.921944,15
4,hko,NaN,1.0,22.301944,114.174167,74


Saved: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig5\publish\wind_obs_stations_20250923_2100.tif
Saved: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig5\publish\wind_obs_stations_20250923_2200.tif
Saved: E:\CityU\Paper Code\06_AI_WRF_UCM\Figs\Fig5\publish\wind_obs_stations_20250923_2300.tif
Done.
